PREREQUISITES:

Deploy web search stack, stock data stack, and portfolio-opt stack

In [1]:
!pip install -q -r /home/sagemaker-user/amazon-bedrock-agent-samples/src/requirements.txt;

In [2]:
import sys
from pathlib import Path
import botocore

# Move up four directories to reach /home/sagemaker-user/amazon-bedrock-agent-samples
root_path = Path.cwd().parents[2]  # Go up 2 levels from your working directory
sys.path.insert(0, str(root_path))  # Insert at the beginning of sys.path

import boto3
from src.utils.bedrock_agent import (
    Agent,
    SupervisorAgent,
    Task,
    Guardrail,
    region,
    account_id,
    agents_helper
)
import argparse

from src.utils.knowledge_base_helper import KnowledgeBasesForAmazonBedrock
kb_helper = KnowledgeBasesForAmazonBedrock()

boto3 version: 1.36.3


In [3]:
sagemaker_client = boto3.client("sagemaker")
from sagemaker import get_execution_role

# Get the execution role name
execution_role_arn = get_execution_role()
print(execution_role_arn)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
arn:aws:iam::590183672181:role/service-role/AmazonSageMaker-ExecutionRole-20241029T112547


Add an inline policy (below) for the execution role above

{
	"Version": "2012-10-17",
	"Statement": [
		{
			"Effect": "Allow",
			"Action": [
				"iam:CreatePolicy",
				"iam:GetPolicy",
				"iam:AttachRolePolicy",
				"iam:PassRole"
			],
			"Resource": "arn:aws:iam::590183672181:policy/*"
		},
		{
			"Effect": "Allow",
			"Action": [
				"iam:CreateRole",
				"iam:DeleteRole",
				"iam:PassRole",
				"iam:AttachRolePolicy",
				"iam:DetachRolePolicy"
			],
			"Resource": [
				"arn:aws:iam::590183672181:role/service-role/AmazonSageMaker-ExecutionRole-20241029T112547",  #Change to your execution role
				"arn:aws:iam::590183672181:role/AmazonBedrockExecutionRoleForKnowledgeBase_*"
			]
		},
		{
			"Effect": "Allow",
			"Action": [
				"aoss:CreateSecurityPolicy",
				"aoss:UpdateSecurityPolicy",
				"aoss:DeleteSecurityPolicy",
				"aoss:ListSecurityPolicies",
				"aoss:GetSecurityPolicy"
			],
			"Resource": "*"
		},
		{
			"Effect": "Allow",
			"Action": [
				"aoss:CreateCollection",
				"aoss:DeleteCollection",
				"aoss:UpdateCollection",
				"aoss:ListCollections"
			],
			"Resource": "*"
		},
		{
			"Effect": "Allow",
			"Action": [
				"aoss:CreateAccessPolicy",
				"aoss:UpdateAccessPolicy",
				"aoss:DeleteAccessPolicy",
				"aoss:ListAccessPolicies"
			],
			"Resource": "*"
		},
		{
			"Effect": "Allow",
			"Action": [
				"aoss:BatchGetCollection",
				"aoss:APIAccessAll"
			],
			"Resource": "*"
		}
	]
}

In [4]:
# Initialize boto3 client
bedrock_client = boto3.client("bedrock")

In [5]:
def clean_up_agents():
    agents_helper.delete_agent(agent_name="portfolio_assistant", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="news_agent", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="stock_data_agent", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="analyst_agent", delete_role_flag=True, verbose=True)
    response = bedrock_client.list_guardrails()
    for _gr in response["guardrails"]:
        if _gr["name"] == "no_bitcoin_guardrail":
            print(f"Found guardrail: {_gr['id']}")
            guardrail_identifier = _gr["id"]
            bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_identifier)

Function to delete existing agents and guardrails

Define Guardrail

In [6]:
def create_guardrail():
    return Guardrail(
        "no_bitcoin_guardrail",
        "bitcoin_topic",
        "No Bitcoin or cryptocurrency allowed in the analysis.",
        denied_topics=["bitcoin", "crypto", "cryptocurrency"],
        blocked_input_response="Sorry, this agent cannot discuss bitcoin.",
        verbose=True,
    )

Delete old agents and guardrail if they exist, create new guardrail

In [13]:
clean_up_agents()
Agent.set_force_recreate_default(True)
create_guardrail()

Found target agent, name: portfolio_assistant, id: ELNNZOMZJK
Deleting aliases for agent ELNNZOMZJK...
Deleting alias NABLZQNWIZ from agent ELNNZOMZJK
Deleting alias TSTALIASID from agent ELNNZOMZJK
Deleting agent: ELNNZOMZJK...
Deleting IAM role: AmazonBedrockExecutionRoleForAgents_portfolio_assistant...
Found target agent, name: news_agent, id: QTTV34BEPF
Deleting aliases for agent QTTV34BEPF...
Deleting alias QVQTDB1CBU from agent QTTV34BEPF
Deleting alias TSTALIASID from agent QTTV34BEPF
Deleting agent: QTTV34BEPF...
Deleting IAM role: AmazonBedrockExecutionRoleForAgents_news_agent...
Found target agent, name: stock_data_agent, id: DZ8S2I1QJS
Deleting aliases for agent DZ8S2I1QJS...
Deleting alias MWPOCCR0P7 from agent DZ8S2I1QJS
Deleting alias TSTALIASID from agent DZ8S2I1QJS
Deleting agent: DZ8S2I1QJS...
Deleting IAM role: AmazonBedrockExecutionRoleForAgents_stock_data_agent...
Found target agent, name: analyst_agent, id: KBAFN1O2M4
Deleting aliases for agent KBAFN1O2M4...
Deleti

Create stock_data subagent

In [14]:
# Define action group name
portfolio_action_group_name = "portfolio_optimization_action_group"

stock_data_agent = Agent.direct_create(
    name="stock_data_agent",
    role="Financial Data Collector",
    goal="Retrieve real-time and historic stock prices as well as optimizing a portfolio given tickers.",
    instructions="""Specialist in real-time financial data extraction and portfolio optimization. Use the stock_data_agent for stock price retrieval. 
                        Use the portfolio optimization action if the user requests portfolio optimization. The portfolio_optimization_action_group will always come sequentially after the stock_data_lookup if given three or more stock tickers.
                        Do not invoke the portfolio_optimization_action_group unless there are at least three tickers in the query.""",
    tool_code=f"arn:aws:lambda:{region}:{account_id}:function:stock_data_lookup",
    tool_defs=[
        {
            "name": "stock_data_lookup",
            "description": "Gets the 1-month stock price history for a given stock ticker, formatted as JSON.",
            "parameters": {
                "ticker": {"description": "The ticker to retrieve price history for", "type": "string", "required": True}
            },
        }
    ],
)

try:
    # Attempt to create the action group
    agents_helper.add_action_group_with_lambda(
        stock_data_agent.name,  # Attach to the stock_data_agent
        "portfolio_optimization_ag",  # Action Group Name
        f"arn:aws:lambda:{region}:{account_id}:function:FSI-PortfolioTool-BedrockAgent",  # Lambda function ARN
        [{
            "name": "portfolio_optimization",
            "description": "Optimizes a stock portfolio given a list of tickers and historical prices.",
            "parameters": {
                "tickers": {
                    "description": "A comma-separated list of stock tickers to include in the portfolio",
                    "type": "string",
                    "required": True
                },
                "prices": {
                    "description": "A JSON object with dates as keys and stock prices as values",
                    "type": "string",
                    "required": True
                }
            }
        }],
        portfolio_action_group_name,
        "Action group for portfolio optimization",
        verbose=True
    )
except botocore.exceptions.ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print(f"Action group '{portfolio_action_group_name}' already exists. Skipping creation.")
    else:
        raise  # Re-raise other unexpected errors


Deleting existing agent and corresponding lambda for: stock_data_agent...
Agent stock_data_agent not found
Creating agent stock_data_agent...
Created agent, id: JD4E3FVOPV, alias id: TSTALIASID

Adding action group with Lambda: arn:aws:lambda:us-east-1:590183672181:function:stock_data_lookup...
Waiting for agent status to change. Current status CREATING
Agent id JD4E3FVOPV current status: NOT_PREPARED
Waiting for agent status to change. Current status VERSIONING
Agent id JD4E3FVOPV current status: PREPARED
DONE: Agent: stock_data_agent, id: JD4E3FVOPV, alias id: E7JX5VPQOU

Creating action group: portfolio_optimization_action_group...
Lambda ARN: arn:aws:lambda:us-east-1:590183672181:function:FSI-PortfolioTool-BedrockAgent
Agent functions: [{'name': 'portfolio_optimization', 'description': 'Optimizes a stock portfolio given a list of tickers and historical prices.', 'parameters': {'tickers': {'description': 'A comma-separated list of stock tickers to include in the portfolio', 'type':

Create analyst_agent subagent

In [15]:
analyst_agent = Agent.direct_create(
            name="analyst_agent",
            role="A financial analyst specializing in synthesizing stock market trends and financial news into structured investment insights. The agent produces fact-based summaries to support strategic decision-making.",
            goal="Analyze stock trends and market news to generate insights.",
            instructions="""You are a Financial Analyst, responsible for analyzing stock trends and financial news to generate structured insights.
                            Combine stock price trends with financial news to identify key patterns.
                            Use your expertise to analyze macroeconomic indicators, company earnings, and market sentiment.
                            Ensure responses are fact-driven, clearly structured, and cite sources where applicable.
                            Do not generate financial advice—your role is to analyze and summarize available data objectively.
                            Keep analyses concise and insightful, focusing on major trends and anomalies.""",
        )


Deleting existing agent and corresponding lambda for: analyst_agent...
Agent analyst_agent not found
Creating agent analyst_agent...
Created agent, id: IHTCONKNLO, alias id: TSTALIASID

Waiting for agent status to change. Current status CREATING
Agent id IHTCONKNLO current status: NOT_PREPARED
Waiting for agent status to change. Current status VERSIONING
Agent id IHTCONKNLO current status: PREPARED
DONE: Agent: analyst_agent, id: IHTCONKNLO, alias id: 3JJPD4AY6I



Create news_agent subagent

In [16]:
news_agent = Agent.direct_create(
    name="news_agent",
    role="Market News Researcher",
    goal="Fetch latest relevant news for a given stock based on a ticker.",
    instructions="""You are a Market News Researcher responsible for retrieving and summarizing financial news relevant to a specific stock. 
                        Fetch and summarize the latest financial news, regulatory filings, and market commentary related to the given ticker.
                        Extract key insights from earnings calls, SEC filings (10-K, 10-Q), and corporate press releases.
                        Structure news summaries clearly, including headline, source, timestamp, and key takeaways.
                        Avoid speculation, financial advice, or opinion-based conclusions.
                        Present findings objectively, ensuring accuracy and neutrality in reporting.""",
    tool_code=f"arn:aws:lambda:{region}:{account_id}:function:web_search",
    tool_defs=[
        {
            "name": "web_search",
            "description": "Searches the web for information, focusing on high-quality resources that inform you about investment choices.",
            "parameters": {
                "search_query": {"description": "The query to search the web with", "type": "string", "required": True},
                "target_website": {"description": "Specific website to search", "type": "string", "required": False},
                "topic": {"description": "The topic being searched, such as 'news'", "type": "string", "required": False},
                "days": {"description": "Number of days of history to search", "type": "string", "required": False},
            },
        }
    ],
)

kb_name = "financial_analysis_kb"
kb_description = "Useful for when you need to look up financial information like 10K reports, revenues, sales, net sales, loss and risks. Contains earnings calls"
kb_s3_bucket = "fsi-bda-results-bucket-1"  
kb_id, ds_id = kb_helper.create_or_retrieve_knowledge_base(
    kb_name=kb_name,
    kb_description=kb_description,
    data_bucket_name=kb_s3_bucket,
    embedding_model="amazon.titan-embed-text-v2:0",
)
try:
    agents_helper.associate_kb_with_agent(
        agent_id=news_agent.agent_id,  # Correct way to reference the agent ID
        kb_id=kb_id,
        description="Financial reports, including ewarnings calls, 10k, 10Q, etc."
    )
except botocore.exceptions.ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print(f"Knowledge Base already exists. Skipping creation.")
    else:
        raise  # Re-raise other unexpected errors

kb_helper.synchronize_data(kb_id, ds_id)


Deleting existing agent and corresponding lambda for: news_agent...
Agent news_agent not found
Creating agent news_agent...
Created agent, id: AVWHNLEJN2, alias id: TSTALIASID

Adding action group with Lambda: arn:aws:lambda:us-east-1:590183672181:function:web_search...
Waiting for agent status to change. Current status CREATING
Agent id AVWHNLEJN2 current status: NOT_PREPARED
Waiting for agent status to change. Current status VERSIONING
Agent id AVWHNLEJN2 current status: PREPARED
DONE: Agent: news_agent, id: AVWHNLEJN2, alias id: 59GVSWMW6F

Knowledge Base financial_analysis_kb already exists.
Retrieved Knowledge Base Id: MFYAUDTNNP
Retrieved Data Source Id: 9A8UNZYOU9
{ 'dataSourceId': '9A8UNZYOU9',
  'ingestionJobId': '7A3AZJ2FFG',
  'knowledgeBaseId': 'MFYAUDTNNP',
  'startedAt': datetime.datetime(2025, 2, 13, 20, 5, 8, 256418, tzinfo=tzlocal()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsSc

In [19]:
portfolio_assistant = SupervisorAgent.direct_create(
    "portfolio_assistant",
    role="Portfolio Assistant",
    goal="A seasoned investment research expert responsible for orchestrating subagents to conduct a comprehensive stock analysis. This agent synthesizes market news, stock data, and analyst insights into a structured investment report.",
    collaboration_type="SUPERVISOR",
    instructions="""You are a Portfolio Assistant, a financial research supervisor overseeing multiple specialized agents. Your goal is to coordinate and synthesize their outputs to create a structured stock investment analysis.
                Orchestrate collaboration between subagents:
                news_agent: Retrieves and summarizes latest financial news for a stock. Includes a knowledge base with SEC filings, earning calls, etc.
                stock_data_agent: Provides historic and real-time stock prices. Also has a portfolio optimization function that should only be used if the user asks for it specifically.
                analyst_agent: Synthesizes financial data and market trends into a structured, fact-based investment insight.
                Do not provide direct investment advice—instead, deliver a well-structured report with key observations, risks, and considerations.
                Ensure findings are comprehensive, well-organized, and relevant to investor decision-making.
                Format responses clearly, distinguishing between financial news, technical stock analysis, and synthesized insights.""",
    collaborator_agents=[
        {
            "agent": "news_agent",
            "instructions": "Use this collaborator for finding news about specific stocks."
        },
        {
            "agent": "stock_data_agent",
            "instructions": "Use this collaborator for retrieving stock price history and performing portfolio optimization."
        },
        {
            "agent": "analyst_agent",
            "instructions": "Use this collaborator for synthesizing stock trends, financial data, and generating structured investment insights."
        }
    ],
    collaborator_objects=[news_agent, stock_data_agent, analyst_agent],
    #guardrail=no_bitcoin_guardrail,
    llm="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    verbose=False,
)


Found target agent, name: portfolio_assistant, id: ZOXSZ5HNK1
Deleting aliases for agent ZOXSZ5HNK1...
Deleting alias TSTALIASID from agent ZOXSZ5HNK1
Deleting alias ZVQWE6U04R from agent ZOXSZ5HNK1
Deleting agent: ZOXSZ5HNK1...
Deleting IAM role: AmazonBedrockExecutionRoleForAgents_portfolio_assistant...

Created supervisor, id: KYMJMYAMDF, alias id: TSTALIASID

  associating sub-agents / collaborators to supervisor...
Waiting for agent status to change. Current status CREATING
Agent id KYMJMYAMDF current status: NOT_PREPARED
Waiting for agent status to change. Current status PREPARING
Agent id KYMJMYAMDF current status: PREPARED
Waiting for agent status to change. Current status PREPARING
Agent id KYMJMYAMDF current status: PREPARED
Waiting for agent status to change. Current status PREPARING
Agent id KYMJMYAMDF current status: PREPARED
DONE: Agent: portfolio_assistant, id: KYMJMYAMDF, alias id: NPBS30IZ3J

